# Shape: generate a domain into lakehouse Delta tables (Python notebook)

Kernel: **Python 3.11 or 3.12** (not PySpark). Generates every table of an installed domain at a
scale preset with a seed, and writes each as a Delta table `Tables/<tablePrefix><table>` in the
default lakehouse. The same seed always generates the same rows.

Next to the tables it writes `Files/<outputDir>/<domain>/contract.json`, the contract that the
domain's own schema implies for these tables (row counts, columns, types, nullability, primary
keys, enumerated values), and `generation.json`. `shape_profile_domain.ipynb` profiles the
tables and checks them against that contract; `shape_generate_gate` runs both from a pipeline.

The exit value is `{domain, scale, seed, mode, tablePrefix, tables, totalRows, contractPath,
kernel}`. `notebookutils.notebook.exit` is the last statement, outside any `try`/`except`.


In [ ]:
%%configure
{"vCores": 8}


In [ ]:
# Upload these wheels to the notebook's resources folder (Resources > builtin):
#   sqllocks_shape-0.9.1-py3-none-any.whl
#   sqllocks_shape_domains-0.9.1-py3-none-any.whl
# They are installed by file path, so a package of the same name and version on PyPI can never be
# installed instead. For the Rust kernel, upload the Linux x86_64 platform wheel
#   sqllocks_shape-0.9.1-cp311-abi3-manylinux_2_28_x86_64.whl
# and put its path in place of the first wheel (the exit value reports the kernel).
# The Python runtime ships numpy 1.26.4 with pandas 2.1.4 and matplotlib 3.8.2, built for numpy 1.
# Shape needs numpy 2, and pandas 2.1.4 cannot load with numpy 2 (pyarrow then fails too), so
# pandas is upgraded to a 2.x built for numpy 2. pip then prints a dependency conflict for
# matplotlib 3.8.2: expected (Shape does not use it; do not plot in this session).
# In a pipeline run inline %pip is off by default; the notebook activity must pass the Boolean
# parameter _inlineInstallationEnabled = true (the shipped pipelines do). A Python notebook
# cannot attach an Environment, and %pip is not supported in High Concurrency mode or in a
# reference run (notebookutils.notebook.run).
%pip install "numpy>=2,<3" "pandas>=2.2.2,<3" builtin/sqllocks_shape-0.9.1-py3-none-any.whl builtin/sqllocks_shape_domains-0.9.1-py3-none-any.whl


In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
domain = "retail"         # an installed domain (`shape list`)
scale = "small"           # a scale preset of the domain (`shape presets`)
seed = 42                 # the same seed always generates the same rows
mode = ""                 # "3nf" or "star"; empty keeps the domain's default schema
tablePrefix = ""          # Delta tables are named <tablePrefix><table>, e.g. "retail_"
writeMode = "overwrite"   # "overwrite" or "append" for the Delta tables
outputDir = "shape"       # the contract and manifest go to Files/<outputDir>/<domain>/


In [ ]:
import json
from pathlib import Path

import shape
from shape.integrations.fabric import generation, onelake
from shape.kernel.dispatch import get_kernel

KERNEL = get_kernel().NAME  # "rust" with a platform wheel, "python" with the pure-Python wheel
print(f"Shape {shape.__version__}, kernel: {KERNEL}")

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"
# A Delta write through the /lakehouse/default/Tables mount fails ("Unable to rename file"):
# in Fabric the tables go to the default lakehouse's OneLake abfss:// URI with the notebook's
# storage token; anywhere else to the folder.
TABLES, STORAGE_OPTIONS = onelake.tables_location(f"{LAKEHOUSE}/Tables")
print("Delta tables go to", TABLES)
MAX_LISTED = 100

# Names reach paths and table names: only identifiers are accepted.
for _label, _value in (("domain", domain), ("scale", scale), ("outputDir", outputDir)):
    generation.check_name(str(_value).strip("/"), _label)
if tablePrefix:
    generation.check_name(str(tablePrefix), "tablePrefix")


In [ ]:
result = generation.generate_domain(
    str(domain),
    scale=str(scale),
    seed=int(seed),
    mode=str(mode) or None,
)
row_counts = {name: table.num_rows for name, table in result.tables.items()}
print(f"Generated {sum(row_counts.values()):,} rows in {len(row_counts)} tables")
for name in result.generation_order:
    print(f"  {name:<24} {row_counts[name]:>10,} rows")


In [ ]:
tables = generation.write_delta_tables(
    result,
    TABLES,
    prefix=str(tablePrefix),
    mode=str(writeMode),
    storage_options=STORAGE_OPTIONS,
)

# The contract the domain's own schema implies for these tables: the pipeline checks the
# profile of the Delta tables against it (shape_profile_domain.ipynb).
# It expects the rows the schema and scale plan, so a short table fails it.
planned = generation.plan_row_counts(str(domain), str(scale), str(mode) or None)
contract = generation.domain_contract(result.schema, planned)
contract_rel = f"{str(outputDir).strip('/')}/{domain}/contract.json"
generation.write_contract(contract, Path(FILES) / contract_rel)
manifest = {
    "domain": domain,
    "scale": scale,
    "seed": int(seed),
    "mode": str(mode) or "default",
    "tablePrefix": str(tablePrefix),
    "tables": tables,
}
(Path(FILES) / contract_rel).with_name("generation.json").write_text(
    json.dumps(manifest, indent=1), encoding="utf-8"
)
print("Delta tables written under", TABLES)
print("Contract written to", Path(FILES) / contract_rel)


In [ ]:
result_value = {
    "domain": domain,
    "scale": scale,
    "seed": int(seed),
    "mode": str(mode) or "default",
    "tablePrefix": str(tablePrefix),
    "tables": tables[:MAX_LISTED],
    "totalRows": sum(t["rows"] for t in tables),
    "contractPath": contract_rel,
    "kernel": KERNEL,
}
print(json.dumps(result_value, indent=2)[:4000])


In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result_value))
